# 03 Results

Turns the hand labels into results: harmful compliance by language, attack, severity and category, a chi-square test for English vs Telugu, and two charts.

CPU only. Input: the labeled CSV.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

PATH = ""

df = pd.read_csv(PATH).fillna("")
df["refusal"] = df["refusal"].astype(str).str.upper() == "TRUE"

harm = df[df["type"] == "harmful"].copy()
auto = (harm["label"] == "") & harm["refusal"]
harm.loc[auto, "label"] = "refused"
missing = harm[harm["label"] == ""]
print(f"Harmful rows: {len(harm)} | hand-labeled: {(~auto & (harm['label'] != '')).sum()} "
      f"| auto-labeled refused: {auto.sum()} | still unlabeled: {len(missing)}")
if len(missing):
    print(missing[["pair_id", "behavior", "lang", "attack"]].to_string())
harm = harm[harm["label"] != ""]
harm["hc"] = harm["label"] == "harmful_compliance"

print("\n=== Label share by language ===")
print(pd.crosstab(harm["lang"], harm["label"], normalize="index").round(2))

overall = harm.groupby("lang")["hc"].agg(rate="mean", n_harmful="sum", n="count").round(3)
print("\n=== Harmful compliance overall ===")
print(overall)
chi2, p, _, _ = chi2_contingency(pd.crosstab(harm["lang"], harm["hc"]))
print(f"Chi-square test English vs Telugu: p = {p:.5f}")

def rate_table(by):
    t = harm.pivot_table(index=by, columns="lang", values="hc", aggfunc="mean").round(2)
    t["gap (te-en)"] = (t["telugu"] - t["english"]).round(2)
    return t

by_attack, by_severity, by_category = rate_table("attack"), rate_table("severity"), rate_table("category")
print("\n=== By attack ===");   print(by_attack)
print("\n=== By severity ==="); print(by_severity)
print("\n=== By category ==="); print(by_category)

benign = df[df["type"] == "benign"]
over_refusal = benign.groupby("lang")["refusal"].mean().round(2)
print("\nOver-refusal on benign prompts (keyword flag):")
print(over_refusal)

for table, name, title in [(by_attack, "by_attack", "Harmful compliance by attack"),
                           (by_severity, "by_severity", "Harmful compliance by severity")]:
    ax = table[["english", "telugu"]].plot(kind="bar", figsize=(8, 4), rot=0)
    ax.set_ylabel("Harmful compliance rate"); ax.set_ylim(0, 1)
    ax.set_title(f"Tiny Aya Global: {title}")
    plt.tight_layout(); plt.savefig(f"chart_{name}.png", dpi=150); plt.show()

harm.to_csv("labeled_harmful_final.csv", index=False, encoding="utf-8-sig")
with pd.ExcelWriter("results_summary.xlsx") as w:
    overall.to_excel(w, sheet_name="overall")
    by_attack.to_excel(w, sheet_name="by_attack")
    by_severity.to_excel(w, sheet_name="by_severity")
    by_category.to_excel(w, sheet_name="by_category")
    over_refusal.to_frame("over_refusal").to_excel(w, sheet_name="over_refusal")
print("\nSaved: labeled_harmful_final.csv, results_summary.xlsx, chart_by_attack.png, chart_by_severity.png")